## Middleware

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

In [3]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [4]:
config = {"configurable":{"thread_id":"test-1"}}

In [5]:
questions = [
    "what is 2+2",
    "what is 2+20",
    "what is 2+21",
    "what is 2+22",
    "what is 2+23",
    "what is 2*0",
    "what is 2*8",
    "what is 2+7"  
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(response)

{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='c33041b8-620e-4a85-b200-e58fec9395ff'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks "what is 2+2". Straightforward. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 77, 'total_tokens': 115, 'completion_time': 0.079409532, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.003005631, 'prompt_tokens_details': None, 'queue_time': 0.372438045, 'total_time': 0.082415163}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca5edfaab2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a036ff-4016-7533-8119-e71cf2d03828-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 38, 'total_tokens': 115, 'output_token_details': {'reasoning': 19}})]}
{'messages': [HumanM

# Human-in-the-Loop Middleware

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
def read_email(email_id:str)-> str:
    """Read an email by its id."""
    return "You have a new email from Rajat."

def send_email(recipent:str,subject:str,body:str) -> str:
    """Send an email."""
    return "Email sent successfully to {recipent} with subject '{subject}'."


In [16]:
agents = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email, send_email],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email": {
                "allowed_decisions": [
                    "approve",
                    "edit",
                    "reject"
                ]
            },
            "read_email": False
        }
    )]
)

In [17]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="send email to rajat@gmail.com with subject 'hello' and body 'how are you'")]},
    config=config
)